In [1]:
import numpy as np

In [2]:
class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_directions(self, E, batch_size=100):
        # Случайное количество фотонов по распределению Пуассона
        n_photons = np.random.poisson(self.Y * E)

        # Генерация фотонов порциями
        for start in range(0, n_photons, batch_size):

            # Размер текущей порции
            n = min(batch_size, n_photons - start)

            # Случайные сферические координаты
            phi = np.random.uniform(0, 2 * np.pi, n)
            cos_theta = np.random.uniform(-1, 1, n)

            sin_theta = np.sqrt(1 - cos_theta**2)

            # Декартовы координаты единичных векторов
            directions = np.column_stack((
                sin_theta * np.cos(phi),
                sin_theta * np.sin(phi),
                cos_theta
            ))

            yield directions

Заменим метод simulate_event() в классе Detector

In [3]:
class Detector:
    def __init__(self, R=2, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

        self.pm_positions = self.generate_positions()

    def generate_positions(self):
        """Равномерное распределение фотоумножителей по сфере."""
        positions = []
        golden_angle = np.pi * (3 - np.sqrt(5))

        for i in range(self.N):
            z = 1 - 2 * (i + 0.5) / self.N
            theta = golden_angle * i
            rho = np.sqrt(1 - z**2)

            x = self.R * rho * np.cos(theta)
            y = self.R * rho * np.sin(theta)
            z = self.R * z

            positions.append([x, y, z])

        return np.array(positions)

    def is_on_detector(self, point, tol=1e-6):
        return abs(np.linalg.norm(point) - self.R) < tol

    def belongs_to_pm(self, point):
        """Проверяет, попадает ли точка в какой-либо фотоумножитель."""
        point = np.asarray(point)

        if not self.is_on_detector(point):
            return False

        cos_angles = self.pm_positions @ point / self.R**2
        cos_angles = np.clip(cos_angles, -1, 1)

        distances = self.R * np.arccos(cos_angles)

        return np.any(distances <= self.r)

    def find_photomultiplier(self, source, direction):
        """
        Определяет, в какой фотоумножитель попал фотон.
        Возвращает индекс фотоумножителя или None.
        """
        source = np.asarray(source)
        direction = np.asarray(direction)

        # Находим пересечение траектории со сферой
        dot = np.dot(source, direction)

        t = -dot + np.sqrt(
            dot**2 + self.R**2 - np.dot(source, source)
        )

        hit_point = source + t * direction

        # Проверяем расстояния до всех фотоумножителей
        cos_angles = self.pm_positions @ hit_point / self.R**2
        cos_angles = np.clip(cos_angles, -1, 1)

        distances = self.R * np.arccos(cos_angles)

        # Индексы фотоумножителей, на которые попал фотон
        hits = np.where(distances <= self.r)[0]

        if len(hits) == 0:
            return None

        return hits[0]

    def generate_photoelectrons(self, n_photons):
        """Разыгрывает количество фотоэлектронов."""
        return np.random.poisson(self.a * n_photons)

    def simulate_event(self, source, E, source_position):
        source_position = np.asarray(source_position)

        n_hits = 0

        cos_limit = np.cos(self.r / self.R)

        # Последовательно обрабатываем порции фотонов
        for directions in source.generate_directions(E):

            # Пересечение лучей с поверхностью сферы
            dot = directions @ source_position

            t = -dot + np.sqrt(
                dot**2 + self.R**2 - np.dot(source_position, source_position)
            )

            hit_points = source_position + t[:, None] * directions

            # Проверка попадания фотонов на фотокатоды
            cos_angles = (hit_points / self.R) @ (
                self.pm_positions / self.R
            ).T

            hits = np.max(cos_angles, axis=1) >= cos_limit

            # Суммируем попадания из всех порций
            n_hits += np.sum(hits)

        # Генерируем суммарное число фотоэлектронов
        return np.random.poisson(self.a * n_hits)
